# Libraries

In [9]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
import statsmodels.api as sm
from statsmodels.graphics.regressionplots import plot_partregress_grid
import statsmodels.formula.api as smf

# Raw data

## price info

In [148]:
df_price = pd.read_csv('./rawdata/amz_daily_mntr_agg1_2.csv')
df_price.sort_values(['collection','zinus_sku','date'],inplace=True)

In [149]:
df_price['fdate'] = [pd.to_datetime(x) for x in df_price.date]
df_price.drop('date',axis=1,inplace=True)

In [150]:
df_price.shape

(118980, 25)

In [151]:
df_price.columns

Index(['prdct_h_lv1', 'prdct_h_lv2', 'collection', 'abc_in', 'parent_asin',
       'zinus_sku', 'asin', 'zinus_sku_nm', 'start_date', 'end_date',
       'bsr_ctgry', 'rank', 'bb_price', 'bb_seller', 'rating', 'rvw_cnt',
       'inv', 'open_po_qty', 'feed_qty', 'bsr_rank_global_min',
       'bsr_rank_global_min_dec', 'bb_amz', 'target_bsr_rnk',
       'target_retail_price', 'fdate'],
      dtype='object')

In [152]:
df_price_tmp = df_price[['fdate','collection','zinus_sku','rank','bb_price','rating','rvw_cnt']].copy()

In [153]:
df_price_tmp.set_index('fdate',inplace=True)

In [154]:
df_price_week = df_price_tmp.groupby(by=['collection','zinus_sku']).resample('W').mean().reset_index().set_index('fdate')

In [155]:
df_price_week[(df_price_week.collection=='HBSM') & (df_price_week.zinus_sku=='SC-HBSM-14Q')]

,collection,zinus_sku,rank,bb_price,rating,rvw_cnt
fdate,,,,,,
2021-12-05,HBSM,SC-HBSM-14Q,28.200000,634.938000,4.5,10469.400000
2021-12-12,HBSM,SC-HBSM-14Q,23.285714,529.834286,4.5,10524.142857
2021-12-19,HBSM,SC-HBSM-14Q,15.142857,446.361429,4.5,10583.142857
2021-12-26,HBSM,SC-HBSM-14Q,17.142857,453.115714,4.5,10638.714286
2022-01-02,HBSM,SC-HBSM-14Q,15.571429,448.038571,4.5,10698.000000
2022-01-09,HBSM,SC-HBSM-14Q,23.714286,444.954286,4.5,10756.142857
2022-01-16,HBSM,SC-HBSM-14Q,25.428571,452.232857,4.5,10812.285714
2022-01-23,HBSM,SC-HBSM-14Q,25.714286,464.382857,4.5,10883.428571
2022-01-30,HBSM,SC-HBSM-14Q,25.142857,480.191429,4.5,10946.142857


In [169]:
tmp_df = df_price_week.reset_index().copy()
df_price_week = tmp_df.set_index(['zinus_sku','fdate'])
df_price_week.index.names = ['sku','date']

df_price_week

In [293]:
df_price_week.shape

(17186, 6)

## sales info

In [87]:
df_sales = pd.read_csv('./rawdata/amz_vc_all_collct_tmp1.csv')
# not sure the week number follows %U (Sunday as the first day of the week)
# or %W (Monday as the first day of the week)
# https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior
df_sales['year_week']=[pd.to_datetime(str(yr_wk)+'0',format="%Y%U%w") for yr_wk in df_sales.yr_wk]
df_sales.drop('yr_wk',axis=1,inplace=True)
df_sales.sort_values(by=['collection','sku','year_week'],inplace=True)

In [88]:
df_sales.set_index(['sku','year_week'],inplace=True)
df_sales.index.names = ['sku','date']

In [89]:
df_sales.shape

(61823, 10)

## rating info (weekly)

In [90]:
df_rating = pd.read_csv('./rawdata/rpa_rvw_weekly_rating.csv')

In [91]:
df_rating['year_week']=[pd.to_datetime(str(yr_wk)+'0',format="%Y%U%w") for yr_wk in df_rating.yr_wk]
df_rating.drop('yr_wk',axis=1,inplace=True)

In [93]:
df_rating.sort_values(by=['collection','sku','year_week'],inplace=True)

In [94]:
df_rating.set_index(['sku','year_week'],inplace=True)
df_rating.index.names=['sku','date']

In [33]:
df_rating.shape

(25597, 25)

## inch info

In [186]:
df_inch = pd.read_csv('./rawdata/erp_sku_profile_noNA.csv')

In [187]:
df_inch.set_index('zinus_sku',inplace=True)
df_inch.index.name='sku'

In [188]:
df_inch.shape

(1519, 1)

## Join tables : price, sales, rating, inch 

In [181]:
df_sales_price = df_sales.join(df_price_week,rsuffix="_FromPrice")

In [182]:
df_all = df_sales_price.join(df_rating, rsuffix="_FromRating")

In [183]:
df_all

asin  \
sku           date                     
ASMPW-10F     2021-06-06  B095NYY1QY   
              2021-06-13  B095NYY1QY   
              2021-06-20  B095NYY1QY   
              2021-06-27  B095NYY1QY   
              2021-07-04  B095NYY1QY   
...                              ...   
ZU-MFMBHD-12T 2022-04-17  B09HZGRBB9   
              2022-04-24  B09HZGRBB9   
              2022-05-01  B09HZGRBB9   
              2022-05-08  B09HZGRBB9   
              2022-05-15  B09HZGRBB9   

                                                                prdct_title  \
sku           date                                                            
ASMPW-10F     2021-06-06                  Zinus Arnav Platform, Full, White   
              2021-06-13                  Zinus Arnav Platform, Full, White   
              2021-06-20  ZINUS Arnav Metal Platform Bed Frame / Wood Sl...   
              2021-06-27  ZINUS Arnav Metal Platform Bed Frame / Wood Sl...   
              2021-07-04  ZINUS Arnav Metal Platform Bed Frame / Wood Sl...   
...                                                                     ...   
ZU-MFMBHD-12T 2022-04-17  ZINUS 12 Inch Cooling Essential Foam Mattress/...   
              2022-04-24  ZINUS 12 Inch Cooling Essential Foam Mattress/...   
              2022-05-01  ZINUS 12 Inch Cooling Essential Foam Mattress/...   
              2022-05-08  ZINUS 12 Inch Cooling Essential Foam Mattress/...   
              2022-05-15  ZINUS 12 Inch Cooling Essential Foam Mattress/...   

                           ord_rev  ord_qty  sale_rank   avg_price      gv  \
sku           date                                                           
ASMPW-10F     2021-06-06   2085.00       15      757.0  139.000000   302.0   
              2021-06-13   4302.00       30      757.0  143.400000   802.0   
              2021-06-20  29400.00      196      488.0  150.000000  7813.0   
              2021-06-27   6000.00       40      535.0  150.000000  1564.0   
              2021-07-04   4500.00       30      388.0  150.000000  1354.0   
...                            ...      ...        ...         ...     ...   
ZU-MFMBHD-12T 2022-04-17   3611.60       19      290.0  190.084211   791.0   
              2022-04-24   4096.34       22      290.0  186.197273   876.0   
              2022-05-01   2588.47       14      351.0  184.890714   570.0   
              2022-05-08   3695.43       20      351.0  184.771500   594.0   
              2022-05-15   6305.44       35      295.0  180.155429   819.0   

                          cnvrsn_rate  rep_oos collection  ...  cnt2 cnt1  \
sku           date                                         ...              
ASMPW-10F     2021-06-06     0.049669      0.0       ASMP  ...   NaN  NaN   
              2021-06-13     0.037406      0.0       ASMP  ...   NaN  NaN   
              2021-06-20     0.025086      0.0       ASMP  ...   NaN  NaN   
              2021-06-27     0.025575      0.0       ASMP  ...   NaN  NaN   
              2021-07-04     0.022157      0.0       ASMP  ...   0.0  0.0   
...                               ...      ...        ...  ...   ...  ...   
ZU-MFMBHD-12T 2022-04-17     0.024020      0.0     MFMBHD  ...   NaN  NaN   
              2022-04-24     0.025114      0.0     MFMBHD  ...   NaN  NaN   
              2022-05-01     0.024561      0.0     MFMBHD  ...   NaN  NaN   
              2022-05-08     0.033670      0.0     MFMBHD  ...   NaN  NaN   
              2022-05-15     0.042735      0.0     MFMBHD  ...   NaN  NaN   

                          cnt0  tot  ratio5  ratio4 ratio3  ratio2  ratio1  \
sku           date                                                           
ASMPW-10F     2021-06-06   NaN  NaN     NaN     NaN    NaN     NaN     NaN   
              2021-06-13   NaN  NaN     NaN     NaN    NaN     NaN     NaN   
              2021-06-20   NaN  NaN     NaN     NaN    NaN     NaN     NaN   
              2021-06-27   NaN  NaN     NaN     NaN    NaN     NaN     NaN  

In [179]:
df_sales.index

MultiIndex([(  'ASMPW-10F', '2021-06-06'),
            (  'ASMPW-10F', '2021-06-13'),
            (  'ASMPW-10F', '2021-06-20'),
            (  'ASMPW-10F', '2021-06-27'),
            (  'ASMPW-10F', '2021-07-04'),
            (  'ASMPW-10F', '2021-07-11'),
            (  'ASMPW-10F', '2021-07-18'),
            (  'ASMPW-10F', '2021-07-25'),
            (  'ASMPW-10F', '2021-08-01'),
            (  'ASMPW-10F', '2021-08-08'),
            ...
            ('SWPBBHW-12T', '2022-03-06'),
            ('SWPBBHW-12T', '2022-03-20'),
            ('SWPBBHW-12T', '2022-03-27'),
            ('SWPBBHW-12T', '2022-04-03'),
            ('SWPBBHW-12T', '2022-04-10'),
            ('SWPBBHW-12T', '2022-04-17'),
            ('SWPBBHW-12T', '2022-04-24'),
            ('SWPBBHW-12T', '2022-05-01'),
            ('SWPBBHW-12T', '2022-05-08'),
            ('SWPBBHW-12T', '2022-05-15')],
           names=['sku', 'date'], length=61823)

In [180]:
df_price_week.index

MultiIndex([(  'ASMPW-10F', '2021-12-05'),
            (  'ASMPW-10F', '2021-12-12'),
            (  'ASMPW-10F', '2021-12-19'),
            (  'ASMPW-10F', '2021-12-26'),
            (  'ASMPW-10F', '2022-01-02'),
            (  'ASMPW-10F', '2022-01-09'),
            (  'ASMPW-10F', '2022-01-16'),
            (  'ASMPW-10F', '2022-01-23'),
            (  'ASMPW-10F', '2022-01-30'),
            (  'ASMPW-10F', '2022-02-06'),
            ...
            ('SWPBBHW-12T', '2022-03-27'),
            ('SWPBBHW-12T', '2022-04-03'),
            ('SWPBBHW-12T', '2022-04-10'),
            ('SWPBBHW-12T', '2022-04-17'),
            ('SWPBBHW-12T', '2022-04-24'),
            ('SWPBBHW-12T', '2022-05-01'),
            ('SWPBBHW-12T', '2022-05-08'),
            ('SWPBBHW-12T', '2022-05-15'),
            ('SWPBBHW-12T', '2022-05-22'),
            ('SWPBBHW-12T', '2022-05-29')],
           names=['sku', 'date'], length=17186)

In [190]:
df_all = df_all.join(df_inch)

In [191]:
df_all.columns

Index(['asin', 'prdct_title', 'ord_rev', 'ord_qty', 'sale_rank', 'avg_price',
       'gv', 'cnvrsn_rate', 'rep_oos', 'collection', 'index',
       'collection_FromPrice', 'rank', 'bb_price', 'rating', 'rvw_cnt',
       'collection_FromRating', 'cnt5', 'cnt4', 'cnt3', 'cnt2', 'cnt1', 'cnt0',
       'tot', 'ratio5', 'ratio4', 'ratio3', 'ratio2', 'ratio1', 'ratio0',
       'inch'],
      dtype='object')

In [192]:
df_all.drop(['index','asin','ord_qty','collection_FromPrice','collection_FromRating'],axis=1,inplace=True)

## NA values

In [199]:
np.sum(df_all.inch.isna())

6676

In [202]:
df_all.shape

(61899, 26)

In [203]:
df_all[~df_all.inch.isna()].shape

(55223, 26)

In [204]:
df_all = df_all[~df_all.inch.isna()]

In [256]:
df_all.shape

(55223, 26)

In [257]:
df_all = df_all.dropna()

In [258]:
df_all.shape

(2898, 26)

In [274]:
a = df_all.index.to_frame()
a.columns = ['SKU','Date']

In [275]:
a

SKU       Date
sku           date                                
ASMPW-10F     2022-04-24      ASMPW-10F 2022-04-24
              2022-05-08      ASMPW-10F 2022-05-08
ASMPW-10K     2022-02-06      ASMPW-10K 2022-02-06
              2022-04-10      ASMPW-10K 2022-04-10
ASMPW-10Q     2022-04-10      ASMPW-10Q 2022-04-10
...                                 ...        ...
ZU-MFMBHD-12Q 2022-03-06  ZU-MFMBHD-12Q 2022-03-06
              2022-03-27  ZU-MFMBHD-12Q 2022-03-27
              2022-05-01  ZU-MFMBHD-12Q 2022-05-01
              2022-05-08  ZU-MFMBHD-12Q 2022-05-08
ZU-MFMBHD-12T 2022-01-30  ZU-MFMBHD-12T 2022-01-30

[2898 rows x 2 columns]

In [276]:
a.reset_index(inplace=True)

In [277]:
a

,sku,date,SKU,Date
0,ASMPW-10F,2022-04-24,ASMPW-10F,2022-04-24
1,ASMPW-10F,2022-05-08,ASMPW-10F,2022-05-08
2,ASMPW-10K,2022-02-06,ASMPW-10K,2022-02-06
3,ASMPW-10K,2022-04-10,ASMPW-10K,2022-04-10
4,ASMPW-10Q,2022-04-10,ASMPW-10Q,2022-04-10
...,...,...,...,...
2893,ZU-MFMBHD-12Q,2022-03-06,ZU-MFMBHD-12Q,2022-03-06
2894,ZU-MFMBHD-12Q,2022-03-27,ZU-MFMBHD-12Q,2022-03-27
2895,ZU-MFMBHD-12Q,2022-05-01,ZU-MFMBHD-12Q,2022-05-01
2896,ZU-MFMBHD-12Q,2022-05-08,ZU-MFMBHD-12Q,2022-05-08


In [282]:
sample_count = a[['SKU','Date']].groupby('SKU').agg('count').sort_values(by='Date',ascending=False)

In [283]:
sample_count.shape

(336, 1)

In [286]:
SKUs = sample_count[sample_count.Date > 20].index

In [287]:
len(SKUs)

32

# Linear model

In [260]:
#metdadata
['collection','sku','inch','prdct_title']

['collection', 'sku', 'inch', 'prdct_title']

In [288]:
predictors = ['Intercept',
              'sale_rank', 
              'avg_price', 
              'gv',
              'cnvrsn_rate',
              'rep_oos', 
              'bb_price', 
              'rating', 
              'rvw_cnt',
              'cnt5', 
              'cnt4', 
              'cnt3', 
              'cnt2', 
              'cnt1', 
              'cnt0', 
              'tot', 
              'ratio5',
              'ratio4', 
              'ratio3', 
              'ratio2', 
              'ratio1', 
              'ratio0']
cols = predictors + ['p_'+x for x in predictors] + ['R2','model']
df_ols_all = pd.DataFrame(columns=cols,index=SKUs)

for sku in SKUs:
    tmp_df_sku = df_all.xs(sku, level='sku', drop_level=True)
    
    model = sm.OLS.from_formula("ord_rev ~ sale_rank+\
                                            avg_price+\
                                            gv+\
                                            cnvrsn_rate+\
                                            rep_oos+\
                                            bb_price+\
                                            rating\
                                            +rvw_cnt\
                                            +cnt5+\
                                            cnt4+\
                                            cnt3+\
                                            cnt2+\
                                            cnt1+\
                                            cnt0+\
                                            tot+\
                                            ratio5\
                                            +ratio4+\
                                            ratio3+\
                                            ratio2+\
                                            ratio1+\
                                            ratio0", data=tmp_df_sku)
    result = model.fit()
    tmp_ols = result.summary().tables[1].data

    coeffs = list()
    p_coeffs = list()
    for k in range(len(predictors)): # Without Intercept
        coeffs.append(float(tmp_ols[k+1][1])) # Coefficients
        p_coeffs.append(float(tmp_ols[k+1][4])) # P > |t|
    coeffs.extend(p_coeffs)
    
    r2 = [float(result.summary().tables[0].data[0][-1])] # R-squrared
    coeffs.extend(r2)
    coeffs.extend([model])
    df_ols_all.loc[sku] = coeffs
    
    print(f"model rank : %d" %model.rank)
    del tmp_df_sku, tmp_ols, result, coeffs, p_coeffs, r2

/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0

model rank : 17
model rank : 18
model rank : 16
model rank : 17
model rank : 17
model rank : 17
model rank : 17
model rank : 17


/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0

model rank : 17
model rank : 18
model rank : 15
model rank : 16
model rank : 16
model rank : 16
model rank : 16
model rank : 16
model rank : 18


/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/wphong/opt/anaconda3/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1918: RuntimeWarning: divide by zero encountered in double_scalars
  return np.sqrt(eigvals[0]/eigvals[-1])


model rank : 16
model rank : 16
model rank : 18
model rank : 16
model rank : 18
model rank : 18
model rank : 17
model rank : 15
model rank : 16
model rank : 15
model rank : 17
model rank : 18
model rank : 15
model rank : 18
model rank : 18


In [292]:
df_ols_all.R2

SKU
OLB-QLSP-14K      0.997
OLB-ABS-9K        0.998
AZ-SBF-07Q          1.0
OLB-SB13-18Q      0.999
OLB-ABS-9Q          1.0
AZ-ASMPH-15Q        1.0
AZ-ASMPH-15K      0.997
AZ-MPSC-14Q       0.999
OLB-SMPB-12T      0.998
AZ-SWFT-150T      0.995
OLB-QLSP-14F      0.991
OLB-ABS-9F        0.997
AZ-MPRC-16K       0.997
AZ-MPRC-16F       0.996
OLB-QLSP-14Q      0.995
OLB-FGM-1200Q     0.993
AZ-SWFT-300K      0.998
OLB-FGM-1000Q     0.996
OLB-MBBF-18Q      0.996
OLB-PWPBBO-12Q    0.999
AZ-MPRC-16Q       0.999
OLB-ABS-5Q          1.0
AZ-SWFT-200Q      0.999
SWPBBHG-12K       0.999
OLB-ABS-9T        0.994
OLB-SMPB-14F      0.999
OLB-PWPBHO-12Q    0.995
OLB-ASB-Q           1.0
AZ-SWFT-150K      0.998
AZ-SBF-07T        0.996
OLB-BNSM-6T       0.985
OLB-GTFM-6T         1.0
Name: R2, dtype: object